<a href="https://colab.research.google.com/github/utkarshsingh171/lunar_landing_reinforcement/blob/main/lunar_landing_deep_q.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gymnasium

In [ ]:
!pip install swig

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 38.2 MB/s eta 0:00:00


In [ ]:
!pip install "gymnasium[box2d]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 100.4 MB/s eta 0:00:00


In [ ]:
import gymnasium as gym
import os, random, numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque


# Initialise the environment
env = gym.make("LunarLander-v3")


state_size = env.observation_space.shape[0]
action_size = env.action_space.n


learning_rate = 5e-4
minibatch = 150
gamma = 0.99
replay_buffer_size = 100000
interpolation_parameter = 1e-3
number_episodes = 5000
max_time_steps = 1000
epsilon_starting_value = 1.0
epsilon_ending_value = 0.01
epsilon_decay_value = 0.995
scores_100_episodes = deque(maxlen=100)


# Neural Network
class ANN(nn.Module):

    def __init__(self, state_size, action_size, seed=42):
        super(ANN, self).__init__()

        self.seed = torch.manual_seed(seed)

        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

    def forward(self, state):
        x = self.fc1(state)
        x = F.relu(x)

        x = self.fc2(x)
        x = F.relu(x)

        return self.fc3(x)


# Replay Memory
class replaymemory(object):

    def __init__(self, capacity):
        self.capacity = capacity
        self.memory = []

    def push(self, event):

        self.memory.append(event)

        if len(self.memory) > self.capacity:
            del self.memory[0]

    def sample(self, batch_size):

        experiences = random.sample(self.memory, batch_size)

        states = torch.from_numpy(
            np.vstack([e[0] for e in experiences if e is not None])
        ).float()

        actions = torch.from_numpy(
            np.vstack([e[1] for e in experiences if e is not None])
        ).long()

        rewards = torch.from_numpy(
            np.vstack([e[2] for e in experiences if e is not None])
        ).float()

        next_states = torch.from_numpy(
            np.vstack([e[3] for e in experiences if e is not None])
        ).float()

        dones = torch.from_numpy(
            np.vstack([e[4] for e in experiences if e is not None])
            .astype(np.uint8)
        ).float()

        return states, actions, rewards, next_states, dones


# Agent
class Agent():

    def __init__(self, state_size, action_size):

        self.state_size = state_size
        self.action_size = action_size

        self.local_ntw = ANN(state_size, action_size)
        self.target_ntw = ANN(state_size, action_size)

        # Initialize target network with local network weights
        self.target_ntw.load_state_dict(self.local_ntw.state_dict())

        self.optimizer = optim.Adam(
            self.local_ntw.parameters(),
            lr=learning_rate
        )

        self.memory = replaymemory(replay_buffer_size)

        self.t_step = 0

    def step(self, state, action, reward, next_state, done):

        self.memory.push(
            (state, action, reward, next_state, done)
        )

        self.t_step = (self.t_step + 1) % 4

        if self.t_step == 0:

            if len(self.memory.memory) > minibatch:

                experiences = self.memory.sample(minibatch)

                self.learn(experiences, gamma)

    def get_action(self, state, epsilon):

        state = torch.from_numpy(state).float().unsqueeze(0)

        self.local_ntw.eval()

        with torch.no_grad():
            action_values = self.local_ntw(state)

        self.local_ntw.train()

        if random.random() > epsilon:

            return np.argmax(
                action_values.cpu().data.numpy()
            )

        else:

            return random.choice(
                np.arange(self.action_size)
            )

    def learn(self, experiences, gamma):

        states, actions, rewards, next_states, dones = experiences

        # Get maximum Q value for next state
        next_q_targets = (
            self.target_ntw(next_states)
            .detach()
            .max(1)[0]
            .unsqueeze(1)
        )

        # Calculate target Q values
        q_targets = rewards + (
            gamma * next_q_targets * (1 - dones)
        )

        # Get expected Q values from local network
        q_expected = self.local_ntw(states).gather(
            1,
            actions
        )

        # Calculate loss
        loss = F.mse_loss(
            q_expected,
            q_targets
        )

        # Optimize local network
        self.optimizer.zero_grad()

        loss.backward()

        self.optimizer.step()

        # Soft update target network
        self.soft_update(
            self.local_ntw,
            self.target_ntw,
            interpolation_parameter
        )

    def soft_update(
        self,
        local_ntw,
        target_ntw,
        interpolation_parameter
    ):

        for target_params, local_params in zip(
            target_ntw.parameters(),
            local_ntw.parameters()
        ):

            target_params.data.copy_(
                interpolation_parameter * local_params.data
                + (1.0 - interpolation_parameter)
                * target_params.data
            )


# Create agent
agent = Agent(state_size, action_size)


# =========================
# TRAINING
# =========================

epsilon = epsilon_starting_value

for episode in range(0, number_episodes):

    state, _ = env.reset()

    score = 0

    for st in range(0, max_time_steps):

        action = agent.get_action(
            state,
            epsilon
        )

        next_state, reward, terminated, truncated, _ = env.step(action)

        done = terminated or truncated

        agent.step(
            state,
            action,
            reward,
            next_state,
            done
        )

        score += reward

        state = next_state

        if done:
            break

    scores_100_episodes.append(score)

    epsilon = max(
        epsilon_ending_value,
        epsilon * epsilon_decay_value
    )

    if episode % 10 == 0:

        print(
            'Episode {} avg score: {:.2f}'.format(
                episode,
                np.mean(scores_100_episodes)
            )
        )

    if np.mean(scores_100_episodes) >= 200:

        print(
            'Congratulations, solved in {:d} episodes \t avg scores: {:.2f}'.format(
                episode,
                np.mean(scores_100_episodes)
            )
        )

        break


env.close()


# =========================
# FIND SUCCESSFUL LANDING
# =========================

import glob
import io
import base64
import imageio
from IPython.display import HTML, display


def show_video_of_model(agent, env_name):

    print("\nSearching for a successful landing...")

    for attempt in range(1, 101):

        env = gym.make(
            env_name,
            render_mode='rgb_array'
        )

        state, _ = env.reset()

        terminated = False
        truncated = False

        frames = []
        score = 0

        while not (terminated or truncated):

            frame = env.render()
            frames.append(frame)

            # No exploration during testing
            action = agent.get_action(
                state,
                0
            )

            state, reward, terminated, truncated, _ = env.step(action)

            score += reward

        env.close()

        print(
            "Test attempt {} | Score: {:.2f}".format(
                attempt,
                score
            )
        )

        # Save ONLY successful landing
        if score >= 200:

            imageio.mimsave(
                'successful_landing.mp4',
                frames,
                fps=30
            )

            print(
                "\nSuccessful landing found!"
            )

            print(
                "Score: {:.2f}".format(score)
            )

            print(
                "Video saved as: successful_landing.mp4"
            )

            return

    print(
        "\nNo successful landing found in 100 attempts."
    )


# Generate video only after successful landing
show_video_of_model(
    agent,
    'LunarLander-v3'
)


# =========================
# DISPLAY VIDEO
# =========================

def show_video():

    mp4list = glob.glob(
        'successful_landing.mp4'
    )

    if len(mp4list) > 0:

        mp4 = mp4list[0]

        video = io.open(
            mp4,
            'r+b'
        ).read()

        encoded = base64.b64encode(video)

        display(
            HTML(
                data=f'''
                <video width="600" controls>
                    <source
                        src="data:video/mp4;base64,{encoded.decode('ascii')}"
                        type="video/mp4">
                </video>
                '''
            )
        )

    else:

        print(
            "Could not find successful_landing.mp4"
        )


show_video()

Episode 0 avg score: -243.74
Episode 10 avg score: -144.43
Episode 20 avg score: -157.55
Episode 30 avg score: -168.62
Episode 40 avg score: -157.53
Episode 50 avg score: -152.26
Episode 60 avg score: -155.21
Episode 70 avg score: -150.12
Episode 80 avg score: -148.55
Episode 90 avg score: -151.01
Episode 100 avg score: -147.71
Episode 110 avg score: -148.55
Episode 120 avg score: -142.12
Episode 130 avg score: -133.99
Episode 140 avg score: -131.42
Episode 150 avg score: -130.76
Episode 160 avg score: -121.88
Episode 170 avg score: -112.40
Episode 180 avg score: -106.90
Episode 190 avg score: -94.78
Episode 200 avg score: -92.03
Episode 210 avg score: -79.82
Episode 220 avg score: -79.10
Episode 230 avg score: -75.21
Episode 240 avg score: -69.62
Episode 250 avg score: -65.65
Episode 260 avg score: -75.55
Episode 270 avg score: -84.77
Episode 280 avg score: -87.65
Episode 290 avg score: -100.68
Episode 300 avg score: -93.84
Episode 310 avg score: -95.09
Episode 320 avg score: -80.50
E


Successful landing found!
Score: 218.81
Video saved as: successful_landing.mp4
